# Wave-action and full-energy fluxes
Plot one frame, several frames, or a pointwise frame average from `fluxes.csv`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'gp2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from gp2d_plotting import (available_frames, curves_for_frames, parameter_float,
    read_csv, read_parameters, repository_root, save_figure, select_frames,
    spectral_abscissa, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
FLUXES_FILE = ROOT / 'output/fluxes.csv'  # CSV produced by the solver.
PARAMETER_FILE = ROOT / 'output/resolved_parameters.txt'  # Supplies c and mu for omega(k).
FIGURE_FILE = ROOT / 'figures/fluxes.pdf'  # Destination for the finished PDF.

# 'single': one curve and exactly one selected frame.
# 'multiple': one curve for every selected frame.
# 'average': one pointwise average over all selected frames.
MODE = 'single'

# Explicit frame numbers to plot. Negative indices count from the end, so [-1]
# means the most recent frame. Set this to None to use START/STOP/STRIDE below.
FRAMES = [-1]
FRAME_START = None  # First frame when FRAMES=None; None means the first available.
FRAME_STOP = None   # Last frame, inclusive; None means the last available.
FRAME_STRIDE = 1    # Keep every nth available frame in the selected range.

# 'wavenumber' plots against k. 'frequency' uses omega=|-c*k^2+mu|.
X_AXIS = 'wavenumber'
# True selects the solver's running segment-mean columns instead of instantaneous data.
USE_SEGMENT_MEAN = False
X_LOG_SCALE = True  # True gives a logarithmic x axis and omits x=0.
# 'linear' shows signed flux directly; 'symlog' resolves small and large signed values.
Y_SCALE = 'linear'
USE_TEX = True      # True uses an external LaTeX installation for all figure text.
FONT_SIZE = 16      # Base font size in points.

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = read_csv(FLUXES_FILE)
frames = select_frames(available_frames(table), FRAMES, start=FRAME_START,
                       stop=FRAME_STOP, stride=FRAME_STRIDE)
wave_column = 'segment_mean_wave_action_flux' if USE_SEGMENT_MEAN else 'wave_action_flux'
energy_column = 'segment_mean_full_energy_flux' if USE_SEGMENT_MEAN else 'full_energy_flux'
k, curves = curves_for_frames(table, frames, [wave_column, energy_column], MODE)
parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
coefficient = parameter_float(parameters, 'dispersionCoefficient', -1.0)
chemical_potential = parameter_float(parameters, 'chemicalPotential', 0.0)
x, x_label = spectral_abscissa(k, X_AXIS, coefficient, chemical_potential)
x_order = np.argsort(x)
mask = np.isfinite(x[x_order]) & ((x[x_order] > 0.0) if X_LOG_SCALE else True)
print(f'Selected frames: {frames}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for curve in curves:
    axes[0].plot(x[x_order][mask], np.asarray(curve[wave_column])[x_order][mask], label=curve['label'])
    axes[1].plot(x[x_order][mask], np.asarray(curve[energy_column])[x_order][mask], label=curve['label'])

coordinate = r'\omega' if X_AXIS == 'frequency' else 'k'
for axis, ylabel in zip(axes, [rf'$\Pi_N({coordinate})$', rf'$\Pi_H({coordinate})$']):
    axis.axhline(0.0, color='0.25', linewidth=0.8)
    axis.set_xlabel(x_label)
    axis.set_ylabel(ylabel)
    if X_LOG_SCALE:
        axis.set_xscale('log')
    axis.set_yscale(Y_SCALE)
    axis.grid(True, which='both', alpha=0.2)
    axis.legend()

saved = save_figure(fig, FIGURE_FILE)
print(f'Wrote {saved}')
plt.show()